# Import mô hình

In [ ]:
import os
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
YOUR_HF_TOKEN = os.environ["HF_TOKEN"]

# 2. Vô hiệu hóa import TensorFlow (nếu chỉ dùng PyTorch)
os.environ["TRANSFORMERS_NO_TF"] = "1"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Nếu có GPU, ưu tiên float16 để tiết kiệm VRAM
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# 3. Load tokenizer với use_auth_token
tokenizer = AutoTokenizer.from_pretrained(
    "Viet-Mistral/Vistral-7B-Chat",
    use_auth_token=YOUR_HF_TOKEN,
)

# 4. Load model với use_auth_token
model = AutoModelForCausalLM.from_pretrained(
    "Viet-Mistral/Vistral-7B-Chat",
    torch_dtype=model_dtype,
    device_map="auto",
    use_auth_token=YOUR_HF_TOKEN,
    low_cpu_mem_usage=True,
)

model.eval()

# 5. Test generation
prompt = "Xin chào, bạn có khỏe không?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.10/site-packages/torch_xla/__init__.py:251: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/transformers/models/auto/tokenization_auto.py:934: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/transformers/models/auto/auto_factory.py:492: FutureWarning: The `use_auth_token` argument is depr

Xin chào, bạn có khỏe không?

Người dùng 2: Tôi đang làm tốt, cảm ơn bạn. Còn bạn thì sao?

Người dùng 1: Tôi cũng ổn, cảm ơn. Tôi đang cố gắng tìm một số thông tin về một số vấn đề kỹ


## import dữ liệu

In [ ]:
import pandas as pd

# Replace 'your_file.csv' with the actual path to your CSV file in Google Drive
csv_path = '/kaggle/input/final-processed-data/final_processed_data.csv'

try:
    ViMUNCH_df = pd.read_csv(csv_path)
    print(f"Successfully loaded CSV from: {csv_path}")
    display(ViMUNCH_df.head())
except FileNotFoundError:
    print(f"Error: File not found at {csv_path}. Please check the path.")
except Exception as e:
    print(f"An error occurred while reading the CSV: {e}")

Successfully loaded CSV from: /kaggle/input/final-processed-data/final_processed_data.csv


,Text
0,Trong ngôn ngữ thiên nhiên Lễ vật là nụ cười
1,Non nước hẹn hò sóng mây dạm ngõ
2,Chim trời phong lưu chim trời Còng gió hào hoa...
3,Giường đá võng cây Chúng mình đi nát sấm tan g...
4,Lồng ngực trái bập bùng như lửa Sóng truyền ch...


## Chia thành 100 set

In [ ]:
import pandas as pd
import math
import os

# Đường dẫn CSV gốc
csv_path = '/content/drive/MyDrive/NCKH/Metaphor_Identification&Classification_LLMs/ViMUNCH_100parts/ViMUNCH_part_1/final_processed_data.csv'

try:
    ViMUNCH_df = pd.read_csv(csv_path)
    print(f"Successfully loaded CSV from: {csv_path}")
    display(ViMUNCH_df.head())

    # ===== Chia thành 100 file =====
    num_parts = 100
    rows_per_part = math.ceil(len(ViMUNCH_df) / num_parts)

    # Tạo thư mục lưu file
    output_dir = "/content/drive/MyDrive/NCKH/Metaphor_Identification&Classification_LLMs/ViMUNCH_100parts/ViMUNCH_part_{i}.json"
    os.makedirs(output_dir, exist_ok=True)

    for i in range(num_parts):
        start_idx = i * rows_per_part
        end_idx = start_idx + rows_per_part
        part_df = ViMUNCH_df.iloc[start_idx:end_idx]

        # Lưu CSV
        csv_file = os.path.join(output_dir, f"ViMUNCH_part_{i+1}.csv")
        part_df.to_csv(csv_file, index=False, encoding='utf-8-sig')

        # Lưu JSON
        json_file = os.path.join(output_dir, f"ViMUNCH_part_{i+1}.json")
        part_df.to_json(json_file, orient='records', force_ascii=False, indent=2)

    print(f"✅ Done! Split into {num_parts} files in folder '{output_dir}'.")

except FileNotFoundError:
    print(f"Error: File not found at {csv_path}. Please check the path.")
except Exception as e:
    print(f"An error occurred while reading the CSV: {e}")

Successfully loaded CSV from: /content/drive/MyDrive/NCKH/Metaphor_Identification&Classification_LLMs/ViMUNCH_100parts/ViMUNCH_part_1/final_processed_data.csv


,Text
0,Trong ngôn ngữ thiên nhiên Lễ vật là nụ cười
1,Non nước hẹn hò sóng mây dạm ngõ
2,Chim trời phong lưu chim trời Còng gió hào hoa...
3,Giường đá võng cây Chúng mình đi nát sấm tan g...
4,Lồng ngực trái bập bùng như lửa Sóng truyền ch...


✅ Done! Split into 100 files in folder '/content/drive/MyDrive/NCKH/Metaphor_Identification&Classification_LLMs/ViMUNCH_100parts/ViMUNCH_part_{i}.json'.


## Lấy 100 mẫu ngẫu nhiên

In [ ]:
# Lấy 100 mẫu ngẫu nhiên
ViMUNCH_sample = ViMUNCH_df.sample(n = 100, random_state = 42) # random_state để kết quả lặp lại

# Đổi tên cột "Text" thành "sentence"
ViMUNCH_sample = ViMUNCH_sample.rename(columns={'Text': 'sentence'})

display(ViMUNCH_sample.head(10))
ViMUNCH_sample_list = ViMUNCH_sample['sentence'].tolist()

,sentence
508306,"Tuy nhiên, giá cả tại chợ không có ưu đãi và t..."
33799,Thôi thì chẳng ở lại làng cùng anh me được thì...
27183,Theo như báo chí thì anh là một con người khá ...
344079,Sự huyền bí của rừng cây Mãi còn là một bậc th...
476313,"Năm nay, đơn vị triển khai chiến dịch gây quỹ ..."
429380,Anh sau đó đã kháng cáo và tiếp tục yêu cầu đư...
193818,Chưa muốn nở: chừng đợi chàng đến vuốt.
5585,vườn cũ đã hoang người cũ đã xa
298456,"Chiêm bao em chết, ôm lòng khóc Tỉnh dậy em cò..."
341061,"Cây bên đường trụi lá, đứng tần ngần,"


## Thực thi

In [ ]:
import torch
import pandas as pd
import json
import time
import sys

# Giả sử model và tokenizer đã được load trước đó
# model = ...
# tokenizer = ...
# model.eval()

VALID_METAPHOR_TYPES = [
    "",
    "Ẩn dụ cấu trúc",
    "Ẩn dụ định hướng",
    "Ẩn dụ bản thể",
    "Ẩn dụ văn hóa dân gian",
    "Ẩn dụ cảm xúc"
]


instruction = """
Bạn là chuyên gia ngôn ngữ học về ẩn dụ trong tiếng Việt. Nhiệm vụ: phân tích 1 câu tiếng Việt theo 3 bước ngắn gọn.

YÊU CẦU (bắt buộc):
1) Xác định câu có chứa ẩn dụ không? → Trả lời **Có** hoặc **Không** (CHỈ 1 từ).
2) Nếu **Có** → Liệt kê các từ/cụm từ chứa ẩn dụ. Nếu ẩn dụ là phép so sánh liền mạch (ví dụ có "như", "là", "tựa như", "giống"), **gộp toàn bộ cụm so sánh** thành một phrase duy nhất.
3) Nếu **Có** → Phân loại ẩn dụ — CHỈ chọn 1 hoặc nhiều từ từ danh sách sau, cách nhau bằng dấu phẩy:
   Ẩn dụ cấu trúc, Ẩn dụ định hướng, Ẩn dụ bản thể, Ẩn dụ văn hóa dân gian, Ẩn dụ cảm xúc

QUY TẮC ĐẦU RA (BẮT BUỘC):
- Chỉ trả 3 dòng **duy nhất**, đúng chính tả và đúng format sau (KHÔNG thêm bất kỳ dòng/chú giải/tiêu đề nào khác):
1. Có/Không: ...
2. Các từ/cụm từ ẩn dụ: [...]
3. Phân loại: [...]
- Nếu mục 1 = "Không" → **để trống** mục 2 và 3 (vẫn giữ 3 dòng).
- Nếu mục 1 = "Có" → mục 2 & 3 **không được để trống**.
- **Ghi chính xác** tên các loại ẩn dụ như bên trên (không viết tắt, không đổi chữ).
- Trả lời bằng **tiếng Việt**. Không giải thích, không mở rộng, không lặp lại câu input.
- Nếu không chắc chắn, chỉ chọn "Có" khi nghĩa bóng rõ ràng (không dùng suy đoán xa).

TÓM NHỎ 5 LOẠI (dùng để tham khảo, *không* in vào kết quả):
- Ẩn dụ cấu trúc: dùng cấu trúc khái niệm này để hiểu khái niệm kia ("Cuộc đời là một sân khấu").
- Ẩn dụ định hướng: dùng phương hướng (lên/xuống, vào/ra) để biểu giá trị trừu tượng ("xuống tinh thần").
- Ẩn dụ bản thể: xem khái niệm trừu tượng như vật thể/sinh thể ("Hồn tôi là một vườn hoa").
- Ẩn dụ văn hóa dân gian: ẩn dụ dựa trên nền tảng văn hóa/dân gian Việt.
- Ẩn dụ cảm xúc: hình ảnh cụ thể diễn tả cảm xúc/tâm trạng.

MẪU (phải trả đúng format như ví dụ):
Input: "Cuộc đời là một sân khấu, và chúng ta đều là những kịch sỹ."
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "sân khấu", "những kịch sỹ"
3. Phân loại: Ẩn dụ cấu trúc

Input: "Cô ấy có mái tóc rất đẹp"
1. Có/Không: Không
2. Các từ/cụm từ ẩn dụ:
3. Phân loại:

Input: "Buồn trông cửa bể chiều hôm, Thuyền ai thấp thoáng cánh buồm xa xa."
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "Buồn trông cửa bể chiều hôm, Thuyền ai thấp thoáng cánh buồm xa xa."
3. Phân loại: Ẩn dụ cảm xúc

CÂU CẦN PHÂN TÍCH:

"""

# =====================
# Hàm post-process robust
# =====================
def postprocess_response(full_prompt, generated_text):
    """
    Clean output by:
    - Removing prompt from generated text if exists
    - Keeping only the first block of response if multiple generations
    - Removing header lines, markdown, extraneous lines
    """
    # Remove prompt prefix if exists
    if generated_text.startswith(full_prompt):
        cleaned = generated_text[len(full_prompt):].strip()
    else:
        cleaned = generated_text.strip()

    # Keep only the first block before model re-generates
    split = cleaned.split("1. ")
    if len(split) > 2:
        cleaned = "1. " + split[1]

    # Remove possible header/markdown lines
    lines = cleaned.splitlines()
    filtered = [line for line in lines if not line.startswith("#") and not line.startswith("---")]
    return "\n".join(filtered).strip()


# =====================
# Hàm validate loại ẩn dụ
# =====================
def validate_analysis_types(analysis_text):
    """
    Ensure only valid metaphor types appear in result
    """
    for vt in VALID_METAPHOR_TYPES:
        if vt.lower() in analysis_text.lower():
            return analysis_text
    return analysis_text + "\nKhông chứa loại ẩn dụ hợp lệ."

# =====================
# Hàm phân tích 1 câu
# =====================
def analyze_metaphor(sentence):
    full_prompt = instruction + f'{sentence}'
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,       # Giới hạn hợp lý để tránh lạc đề
            temperature= 0,          # THẤP hơn nữa để đảm bảo chắc chắn, ít ngẫu nhiên
            top_p= 1,                # Giảm nhẹ để tránh mẫu hiếm, tăng ổn định
            do_sample=False,          # Tắt lấy mẫu → deterministic, rất phù hợp bài toán structured
            num_return_sequences=1,
            repetition_penalty=1.1    # Tăng nhẹ để giảm lặp từ, đặc biệt khi sinh số thứ tự 1., 2., 3.
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    clean_text = postprocess_response(full_prompt, generated_text)
    validated_text = validate_analysis_types(clean_text)
    return validated_text



# =====================
# Pipeline phân tích list câu part1
# =====================
ViMUNCH_sample_part1 = pd.read_csv('/kaggle/working/ViMUNCH_100_parts/ViMUNCH_part_1.csv')

# Đổi tên cột "Text" thành "sentence"
ViMUNCH_sample_part1 = ViMUNCH_sample_part1.rename(columns={'Text': 'sentence'})

display(ViMUNCH_sample_part1.head(10))

ViMUNCH_sample_part1_list = ViMUNCH_sample_part1['sentence'].tolist()
sentences = ViMUNCH_sample_part1_list[600:]
results = []

# =====================
# Parse kết quả
# =====================

import re

def parse_analysis(text):
    """
    Parse kết quả trả về từ model thành dict cấu trúc.
    Hỗ trợ 2 format:
    (1)
    1. Có/Không
    - Có
    2. Các từ/cụm từ ẩn dụ:
    - "...."
    - "...."
    3. Phân loại:
    - Ẩn dụ cấu trúc
    - Ẩn dụ cảm xúc

    (2)
    1. Có/Không: Có
    2. Các từ/cụm từ ẩn dụ: "..."
    3. Phân loại: Ẩn dụ ...
    """
    lines = text.strip().split("\n")

    has_metaphor = ""
    metaphor_phrases = []
    metaphor_types = []

    current_section = None

    for line in lines:
        line = line.strip()

        if line.startswith("1."):
            current_section = "has_metaphor"
            # Format (2): 1. Có/Không: Có
            if ":" in line:
                parts = line.split(":")
                if len(parts) > 1:
                    has_metaphor = parts[1].strip()
                    if has_metaphor.lower() == "không":
                        # Clean output for "Không"
                        metaphor_phrases = []
                        metaphor_types = []
                        return {
                            "has_metaphor": "Không",
                            "metaphor_phrases": metaphor_phrases,
                            "metaphor_types": metaphor_types
                        }
        elif line.startswith("2."):
            current_section = "metaphor_phrases"
            # Format (2): 2. Các từ/cụm từ ẩn dụ: "...."
            if ":" in line:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    content = parts[1].strip()
                    if content:
                        items = [c.strip().strip('"') for c in content.split(",") if c.strip()]
                        metaphor_phrases.extend(items)
        elif line.startswith("3."):
            current_section = "metaphor_types"
            # Format (2): 3. Phân loại: Ẩn dụ bản thể
            if ":" in line:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    content = parts[1].strip()
                    if content:
                        items = [c.strip() for c in content.split(",") if c.strip()]
                        metaphor_types.extend(items)

        elif line.startswith("-"):
            content = line[1:].strip().strip('"')
            if current_section == "has_metaphor":
                has_metaphor = content
                if has_metaphor.lower() == "không":
                    # Clean output for "Không"
                    metaphor_phrases = []
                    metaphor_types = []
                    return {
                        "has_metaphor": "Không",
                        "metaphor_phrases": metaphor_phrases,
                        "metaphor_types": metaphor_types
                    }
            elif current_section == "metaphor_phrases":
                metaphor_phrases.append(content)
            elif current_section == "metaphor_types":
                metaphor_types.append(content)

    # Final clean: if has_metaphor empty ➔ set to "Không"
    if not has_metaphor:
        has_metaphor = "Không"
        metaphor_phrases = []
        metaphor_types = []

    # Clean placeholder "[...]"
    if metaphor_phrases == ["[...]"]:
        metaphor_phrases = []
    if metaphor_types == ["[...]"]:
        metaphor_types = []

    return {
        "has_metaphor": has_metaphor,
        "metaphor_phrases": metaphor_phrases,
        "metaphor_types": metaphor_types
    }


'''# Phân tích
for idx, s in enumerate(sentences, 1):
    print(f"Đang phân tích câu {idx}/{len(sentences)}...")
    print(s)
    res = analyze_metaphor(s)
    if res == '':
      print('Không có ẩn dụ!')
    else:
      print(res)
    parsed = parse_analysis(res)
    results.append({
        "sentence": s,
        "has_metaphor": parsed["has_metaphor"],
        "metaphor_phrases": parsed["metaphor_phrases"],
        "metaphor_types": parsed["metaphor_types"]
    })

    # Keep alive log
    print(f"Câu {idx} phân tích xong lúc {time.strftime('%H:%M:%S')}")
    sys.stdout.flush()  # Đảm bảo in ra ngay lập tức

    time.sleep(5)

with open("100randomsamples_metaphor_analysis_parsed (4).json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Đã lưu kết quả parse vào '100randomsamples_metaphor_analysis_parsed (4).json'.")

# =====================
# Xuất JSON
# =====================
with open("/kaggle/working/100randomsamples_metaphor_analysis_parsed (4).json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Đã lưu kết quả parse vào '100randomsamples_metaphor_analysis_parsed (4).json'.")

# =====================
# Xuất CSV
# =====================
df = pd.DataFrame(results)
df.to_csv("/kaggle/working/100randomsamples_metaphor_analysis_results (4).csv", index=False, encoding="utf-8-sig")
print("Đã lưu kết quả vào '100randomsamples_metaphor_analysis_results (4).csv'.")'''

import time
import sys
import json
import pandas as pd
import math

batch_size = 100
total_batches = math.ceil(len(sentences) / batch_size)

for batch_idx in range(total_batches):
    start_idx = batch_idx * batch_size
    end_idx = min((batch_idx + 1) * batch_size, len(sentences))
    batch_sentences = sentences[start_idx:end_idx]

    results = []

    print(f"\n=== Bắt đầu phân tích batch {batch_idx+1}/{total_batches} ===")

    for idx, s in enumerate(batch_sentences, 1):
        print(f"Đang phân tích câu {start_idx + idx}/{len(sentences)}...")
        print(s)
        res = analyze_metaphor(s)
        if res == '':
            print('Không có ẩn dụ!')
        else:
            print(res)
        parsed = parse_analysis(res)
        results.append({
            "sentence": s,
            "has_metaphor": parsed["has_metaphor"],
            "metaphor_phrases": parsed["metaphor_phrases"],
            "metaphor_types": parsed["metaphor_types"]
        })

        # Keep alive log
        print(f"Câu {start_idx + idx} phân tích xong lúc {time.strftime('%H:%M:%S')}")
        sys.stdout.flush()
        time.sleep(5)

    # Lưu kết quả của batch
    json_path = f"/kaggle/working/metaphor_analysis_part1_batch_{batch_idx+7}.json"
    csv_path = f"/kaggle/working/metaphor_analysis_part1_batch_{batch_idx+7}.csv"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"✅ Đã lưu file JSON: {json_path}")

    df = pd.DataFrame(results)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"✅ Đã lưu file CSV: {csv_path}")

,sentence
0,Trong ngôn ngữ thiên nhiên Lễ vật là nụ cười
1,Non nước hẹn hò sóng mây dạm ngõ
2,Chim trời phong lưu chim trời Còng gió hào hoa...
3,Giường đá võng cây Chúng mình đi nát sấm tan g...
4,Lồng ngực trái bập bùng như lửa Sóng truyền ch...
5,Tín hiệu của tự do hoang dã Anh phải gỡ xuống ...
6,Và rút khỏi bấy nhiêu lòe loẹt dép giày
7,Để đến với em Chân đất đầu trần
8,Bói cá đọc tiểu thuyết đại dương theo chương h...
9,Nai sóc đọc sử thi non ngàn bằng diễn ngôn quả nụ



=== Bắt đầu phân tích batch 1/57 ===
Đang phân tích câu 1/5682...
Con ơi, con sẽ phải sống trong rừng sâu... lớn lên trong rừng sâu... con sẽ sống tách ra khỏi xã hội... con sẽ trở thành người rừng sống ở hang ở hố như con beo con gấu, dốt nát không biết chữ biết nghĩa là gì... rồi ma thiêng nước độc... đói rét, bệnh tật... con ơi, con còn bé liệu con có chịu đựng nổi không?


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Đáp án mẫu:
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "rừng sâu", "người rừng", "hang ở hố", "con beo con gấu", "dốt nát", "ma thiêng nước độc", "đói rét", "bệnh tật".
3. Phân loại: Ẩn dụ văn hóa dân gian
Câu 1 phân tích xong lúc 14:46:42
Đang phân tích câu 2/5682...
Con tôi đã ngủ say.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Tôi cứ nghĩ mãi về chuyện đã xảy ra ngày hôm nay.
Câu 13 phân tích xong lúc 16:15:15
Đang phân tích câu 14/5682...
Tôi đang nghĩ đến những kế hoạch phải làm ngày mai.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Không có ẩn dụ!
Câu 14 phân tích xong lúc 16:15:42
Đang phân tích câu 15/5682...
Tôi đã tìm thấy đúng con đường tôi đi.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


KeyboardInterrupt: 

In [ ]:
import time
for i in range(180):
    print(f"Keep alive {i}")
    time.sleep(60)  # in mỗi phút một lần

In [1]:
from transformers import AutoTokenizer
tokens = tokenizer(instruction)["input_ids"]
print("Token count:", len(tokens))

KeyboardInterrupt: 

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import json
import os

# Thư mục chứa file json gốc
input_folder = "/content/drive/MyDrive/NCKH/Metaphor_Identification&Classification_LLMs/ViMUNCH_100parts/ViMUNCH_part_3/metaphor_analysis_part3_batch_{i}.json"
# Thư mục lưu kết quả
output_folder = "/content/drive/MyDrive/NCKH/Metaphor_Identification&Classification_LLMs/ViMUNCH_100parts/ViMUNCH_part_3/metaphor_analysis_part3_with_offsets_batch_{i}.json"
os.makedirs(output_folder, exist_ok=True)

# Lặp qua tất cả file .json trong thư mục input
num_metaphor_sentences = 0
for filename in os.listdir(input_folder):
    if filename.lower().endswith(".json"):
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)

        # Đọc dữ liệu gốc
        with open(input_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Chuẩn hoá nhãn
        for entry in data:
            if entry.get("has_metaphor") == "Có":
                entry["has_metaphor"] = 1
            elif entry.get("has_metaphor") == "Không":
                entry["has_metaphor"] = 0

        filtered_data = []  # chỉ giữ câu có ẩn dụ

        for entry in data:
            if entry.get("has_metaphor") == 1 and entry.get("metaphor_phrases"):
                sentence = entry["sentence"]
                updated_phrases = []
                for phrase in entry["metaphor_phrases"]:
                  if isinstance(sentence, str):
                      start = sentence.find(phrase)
                      if start != -1:
                          end = start + len(phrase)
                          updated_phrases.append([phrase, start, end - 1])
                entry["metaphor_phrases"] = updated_phrases
            filtered_data.append(entry)
            # nếu has_metaphor = 0 thì bỏ qua, không append
            '''if entry.get("metaphor_types"):
              eng_metaphor_types = []
              for metaphor_type in entry.get("metaphor_types"):
                if metaphor_type == "Ẩn dụ cấu trúc":
                  eng_metaphor_types.append("Structural metaphor")
                if metaphor_type == "Ẩn dụ định hướng":
                  eng_metaphor_types.append("Orientational metaphor")
                if metaphor_type == "Ẩn dụ bản thể":
                  eng_metaphor_types.append("Ontological metaphor")
                if metaphor_type == "Ẩn dụ văn hóa dân gian":
                  eng_metaphor_types.append("Folk-cultural metaphor")
                if metaphor_type == "Ẩn dụ cảm xúc":
                  eng_metaphor_types.append("Emotional metaphor")
              entry["metaphor_types"] = eng_metaphor_types
            else:
              entry["metaphor_types"] = []'''
        # Ghi file kết quả
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(filtered_data, f, ensure_ascii=False, indent=2)

print("Đã xử lý xong tất cả file trong thư mục:", input_folder)

Đã xử lý xong tất cả file trong thư mục: /content/drive/MyDrive/NCKH/Metaphor_Identification&Classification_LLMs/ViMUNCH_100parts/ViMUNCH_part_3/metaphor_analysis_part3_batch_{i}.json


# LLaMA-2-13B-chat

## import

In [ ]:
import os
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
YOUR_HF_TOKEN = os.environ["HF_TOKEN"]

# Vô hiệu hóa import TensorFlow (nếu chỉ dùng PyTorch)
os.environ["TRANSFORMERS_NO_TF"] = "1"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Nếu có GPU, ưu tiên float16 để tiết kiệm VRAM
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# 1. Load tokenizer với use_auth_token
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf",
    use_auth_token=YOUR_HF_TOKEN,
)

# 2. Load model với use_auth_token
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf",
    torch_dtype=model_dtype,
    device_map="auto",
    use_auth_token=YOUR_HF_TOKEN,
    low_cpu_mem_usage=True,
)

model.eval()

# 3. Test generation
prompt = "Xin chào, bạn có khỏe không?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
while True:
    # 1. Nhập prompt từ người dùng
    prompt = input("Nhập prompt (hoặc gõ 'exit' để thoát): ")

    # 2. Thoát nếu nhập 'exit'
    if prompt.lower() == 'exit':
        print("Kết thúc.")
        break

    # 3. Tokenize input và chuyển sang device của model
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 4. Generate output (không cần tính gradient, tiết kiệm RAM)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            temperature=0.2,
            top_p=0.8,
            do_sample=True,
            num_return_sequences=1
        )

    # 5. Decode output
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 6. In kết quả
    print("\n=== Trả lời ===")
    print(response)
    print("================\n")

In [ ]:
import torch
import pandas as pd
import json
import time

VALID_METAPHOR_TYPES = [
    "",
    "Ẩn dụ cấu trúc",
    "Ẩn dụ định hướng",
    "Ẩn dụ bản thể",
    "Ẩn dụ văn hóa dân gian",
    "Ẩn dụ cảm xúc"
]

instruction = """
Bạn là chuyên gia ngôn ngữ học ẩn dụ trong tiếng Việt. Hãy phân tích câu tiếng Việt được cung cấp.

### NHIỆM VỤ

1. Xác định ẩn dụ
- Câu có chứa phép ẩn dụ không?
→ Trả lời Có hoặc Không (CHỈ 1 từ duy nhất).

2. Liệt kê từ/cụm từ ẩn dụ (chỉ trả lời nếu câu có ẩn dụ)
- Các từ/cụm từ chứa phép ẩn dụ: [...]
- Nếu ẩn dụ nằm ở cả câu → ghi "Cả câu".
- Nếu ẩn dụ là một phép so sánh liền mạch, hãy gộp cả câu so sánh thành một cụm phrase duy nhất.

3. Phân loại ẩn dụ (chỉ trả lời nếu câu có ẩn dụ)
- Phân loại: [...]
- CHỈ được chọn trong 5 loại bên dưới, cách nhau bằng dấu "," nếu có nhiều loại.
- KHÔNG tự nghĩ thêm loại mới. KHÔNG giải thích.
- Mỗi loại KHÔNG CẦN đặt trong dấu "..."
---

**5 loại ẩn dụ (Chỉ chọn các loại này, không thêm loại khác, không viết sai tên loại ẩn dụ)**:
1. Ẩn dụ cấu trúc
2. Ẩn dụ định hướng
3. Ẩn dụ bản thể
4. Ẩn dụ văn hóa dân gian
5. Ẩn dụ cảm xúc

### CÁC LOẠI ẨN DỤ (ĐỊNH NGHĨA + VÍ DỤ)

1. Ẩn dụ cấu trúc (Structural metaphor)
- Định nghĩa: Một lĩnh vực khái niệm được cấu trúc hóa theo một lĩnh vực khác, giúp người đọc hiểu khái niệm phức tạp thông qua khái niệm quen thuộc.
- Ví dụ:
  • “Cuộc đời là một sân khấu, và chúng ta đều là những kịch sỹ.”
  • “Người với người sống để yêu nhau.”

2. Ẩn dụ định hướng (Orientational metaphor)
- Định nghĩa: Diễn tả các khái niệm trừu tượng thông qua phương hướng không gian (lên/xuống, vào/ra, trước/sau), giúp định hình giá trị tích cực hay tiêu cực.
- Ví dụ:
  • “Lên non mới biết non cao.”
  • “Dấn thân vô là phải chịu tù đày.”

3. Ẩn dụ bản thể (Ontological metaphor)
- Định nghĩa: Xem các khái niệm trừu tượng như vật thể hữu hình hoặc sinh thể, giúp dễ dàng hình dung và tác động đến.
- Ví dụ:
  • “Hồn tôi là một vườn hoa lá.”
  • “Thân em vừa trắng lại vừa tròn.”

4. Ẩn dụ văn hóa dân gian (Cultural metaphor)
- Định nghĩa: Ẩn dụ bắt nguồn từ kinh nghiệm tập thể, truyền thống và văn hóa dân gian, từ đó phản ánh sâu sắc thế giới quan và kinh nghiệm sống của cộng đồng người Việt, tạo nên bản sắc riêng trong ngôn ngữ.

- Ví dụ:
  • “Thân em như hạt mưa sa.”
  • “Đất là nơi Chim về, Nước là nơi Rồng ở.”

5. Ẩn dụ cảm xúc (Emotion metaphor)
- Định nghĩa: Biểu đạt tinh tế trạng thái tình cảm, cảm xúc con người thông qua những hình ảnh cụ thể, giàu sức gợi. Đây cũng là nét đặc trưng nổi bật của tiếng Việt— giàu tính biểu cảm.
- Ví dụ:
  • “Tháng Giêng ngon như một cặp môi gần.”

---

### YÊU CẦU CHO CÂU TRẢ LỜI

**BẮT BUỘC** tuân thủ cấu trúc **chính xác** dưới đây:
1. Có/Không
2. Các từ/cụm từ ẩn dụ: [...]
3. Phân loại: [...]

LƯU Ý **(đọc kĩ phần này và tuyệt đối tuân theo)**:
- TRẢ LỜI THEO ĐÚNG CẤU TRÚC ĐÃ NÊU.
- Nếu "Không" ở mục 1 ➔ Dừng, KHÔNG trả lời mục 2, 3.
- Nếu mục 1 trả lời "Có", BẮT BUỘC mục 2 và 3 phải có nội dung.
- KHÔNG GIẢI THÍCH THÊM. KHÔNG viết thêm nhận xét.
- KHÔNG CẦN VIẾT LẠI CÂU HỎI, KHÔNG CẦN PHÂN TÍCH, CHỈ VIẾT CÂU TRẢ LỜI NGẮN GỌN THEO CẤU TRÚC BÊN DƯỚI.
- KHÔNG VIẾT BẤT KỲ TIÊU ĐỀ, HEADER, EMOJI, KÝ HIỆU ĐẶC BIỆT NÀO.
- CHỈ VIẾT NỘI DUNG CÂU TRẢ LỜI THEO ĐÚNG CẤU TRÚC, KHÔNG CÓ DÒNG NÀO KHÁC.
- CHỈ TRẢ LỜI 1 LẦN DUY NHẤT CHO MỖI CÂU.
- "Cấu trúc ẩn dụ" -> "Ẩn dụ cấu trúc"
- KHÔNG THÊM HAY VIẾT LẠI BẤT CỨ NỘI DUNG NÀO KHÁC
- CÓ thể có NHIỀU LOẠI ẨN DỤ, hãy liệt kê đầy đủ tất cả các loại liên quan.
- Nếu ẩn dụ là một phép so sánh liền mạch, hãy gộp cả câu so sánh thành một cụm phrase.
- Nếu câu chứa phép SO SÁNH (ví dụ: "như", "như là") ➔ nhận diện là ẩn dụ cấu trúc (nếu phép so sánh chuyển nghĩa).
- Nếu không tìm thấy từ/cụm từ hoặc phân loại rõ ràng, trả lời "Cả câu" cho mục 2 và chọn loại ẩn dụ phù hợp nhất cho mục 3, không để trống.

Ví dụ 1:
Câu: “Cuộc đời là một sân khấu, và chúng ta đều là những kịch sỹ.”
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "sân khấu”, "những kịch sỹ"
3. Phân loại: Ẩn dụ cấu trúc

Ví dụ 2:
Câu: “Lên non mới biết non cao.”
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: “Lên non”
3. Phân loại: Ẩn dụ định hướng

Ví dụ 3:
Câu: “Hồn tôi là một vườn hoa lá.”
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: “vườn hoa lá”
3. Phân loại: Ẩn dụ bản thể

Ví dụ 4:
Câu: “Đất là nơi Chim về, Nước là nơi Rồng ở.”
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: “nơi Chim về”, “nơi Rồng ở”
3. Phân loại: Ẩn dụ văn hóa dân gian

Ví dụ 5:
Câu: “Tháng Giêng ngon như một cặp môi gần.”
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "ngon", "một cặp môi gần"
3. Phân loại: Ẩn dụ bản thể, Ẩn dụ cảm xúc

Ví dụ 6:
Câu: "Buồn trông cửa bể chiều hôm, Thuyền ai thấp thoáng cánh buồm xa xa."
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "Cả câu"
3. Phân loại: Ẩn dụ cảm xúc

Ví dụ 7:
Câu: "Cô ấy có mái tóc rất đẹp"
1. Có/Không: Không
2. Các từ/cụm từ ẩn dụ:
3. Phân loại:

Ví dụ 8:
Câu: “Cuộc sống là một chuyến tàu dài.”
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "chuyến tàu dài"
3. Phân loại: Ẩn dụ cấu trúc

Ví dụ 9:
Câu: “Anh ấy đang xuống tinh thần.”
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "xuống tinh thần"
3. Phân loại: Ẩn dụ định hướng

Ví dụ 10:
Câu: “Nỗi buồn này thật nặng nề.”
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "nặng nề"
3. Phân loại: Ẩn dụ bản thể

Ví dụ 11:
Câu: “Thân em như củ ấu gai.”
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "Thân em", "củ ấu gai"
3. Phân loại: Ẩn dụ văn hóa dân gian

Ví dụ 12:
Câu: “Tình yêu ấy ngọt ngào như mật.”
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "ngọt ngào", "mật"
3. Phân loại: Ẩn dụ cảm xúc

Ví dụ 13:
Câu: “Hôm nay trời nắng đẹp.”
1. Có/Không: Không
2. Các từ/cụm từ ẩn dụ:
3. Phân loại:

Ví dụ 14:
Câu: “Anh là mặt trời sưởi ấm cuộc đời em.”
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "mặt trời sưởi ấm"
3. Phân loại: Ẩn dụ cấu trúc, Ẩn dụ cảm xúc

Ví dụ 15:
Câu: “Cô ấy rơi vào tuyệt vọng.”
1. Có/Không: Có
2. Các từ/cụm từ ẩn dụ: "rơi vào tuyệt vọng"
3. Phân loại: Ẩn dụ định hướng, Ẩn dụ bản thể
---

### CÂU CẦN PHÂN TÍCH:

"""

# =====================
# Hàm post-process robust
# =====================
def postprocess_response(full_prompt, generated_text):
    """
    Clean output by:
    - Removing prompt from generated text if exists
    - Keeping only the first block of response if multiple generations
    - Removing header lines, markdown, extraneous lines
    """
    # Remove prompt prefix if exists
    if generated_text.startswith(full_prompt):
        cleaned = generated_text[len(full_prompt):].strip()
    else:
        cleaned = generated_text.strip()

    # Keep only the first block before model re-generates
    split = cleaned.split("1. ")
    if len(split) > 2:
        cleaned = "1. " + split[1]

    # Remove possible header/markdown lines
    lines = cleaned.splitlines()
    filtered = [line for line in lines if not line.startswith("#") and not line.startswith("---")]
    return "\n".join(filtered).strip()

# =====================
# Hàm validate loại ẩn dụ
# =====================
def validate_analysis_types(analysis_text):
    """
    Ensure only valid metaphor types appear in result
    """
    for vt in VALID_METAPHOR_TYPES:
        if vt.lower() in analysis_text.lower():
            return analysis_text
    return analysis_text + "\nKhông chứa loại ẩn dụ hợp lệ."

# =====================
# Hàm phân tích 1 câu
# =====================
def analyze_metaphor(sentence):
    full_prompt = instruction + f'{sentence}'
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            temperature=0.2,
            top_p=0.8,
            do_sample=True,
            num_return_sequences=1
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    clean_text = postprocess_response(full_prompt, generated_text)
    validated_text = validate_analysis_types(clean_text)
    return validated_text

# =====================
# Pipeline phân tích list câu
# =====================
"""sentences = ViMUNCH_sample_list"""
sentences = ['Người với người sống để yêu nhau.']
results = []

# =====================
# Parse kết quả
# =====================

import re

def parse_analysis(text):
    """
    Parse kết quả trả về từ model thành dict cấu trúc.
    Hỗ trợ 2 format:
    (1)
    1. Có/Không
    - Có
    2. Các từ/cụm từ ẩn dụ:
    - "...."
    - "...."
    3. Phân loại:
    - Ẩn dụ cấu trúc
    - Ẩn dụ cảm xúc

    (2)
    1. Có/Không: Có
    2. Các từ/cụm từ ẩn dụ: "..."
    3. Phân loại: Ẩn dụ ...
    """
    lines = text.strip().split("\n")

    has_metaphor = ""
    metaphor_phrases = []
    metaphor_types = []

    current_section = None

    for line in lines:
        line = line.strip()

        if line.startswith("1."):
            current_section = "has_metaphor"
            # Format (2): 1. Có/Không: Có
            if ":" in line:
                parts = line.split(":")
                if len(parts) > 1:
                    has_metaphor = parts[1].strip()
                    if has_metaphor.lower() == "không":
                        # Clean output for "Không"
                        metaphor_phrases = []
                        metaphor_types = []
                        return {
                            "has_metaphor": "Không",
                            "metaphor_phrases": metaphor_phrases,
                            "metaphor_types": metaphor_types
                        }
        elif line.startswith("2."):
            current_section = "metaphor_phrases"
            # Format (2): 2. Các từ/cụm từ ẩn dụ: "...."
            if ":" in line:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    content = parts[1].strip()
                    if content:
                        items = [c.strip().strip('"') for c in content.split(",") if c.strip()]
                        metaphor_phrases.extend(items)
        elif line.startswith("3."):
            current_section = "metaphor_types"
            # Format (2): 3. Phân loại: Ẩn dụ bản thể
            if ":" in line:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    content = parts[1].strip()
                    if content:
                        items = [c.strip() for c in content.split(",") if c.strip()]
                        metaphor_types.extend(items)

        elif line.startswith("-"):
            content = line[1:].strip().strip('"')
            if current_section == "has_metaphor":
                has_metaphor = content
                if has_metaphor.lower() == "không":
                    # Clean output for "Không"
                    metaphor_phrases = []
                    metaphor_types = []
                    return {
                        "has_metaphor": "Không",
                        "metaphor_phrases": metaphor_phrases,
                        "metaphor_types": metaphor_types
                    }
            elif current_section == "metaphor_phrases":
                metaphor_phrases.append(content)
            elif current_section == "metaphor_types":
                metaphor_types.append(content)

    # Final clean: if has_metaphor empty ➔ set to "Không"
    if not has_metaphor:
        has_metaphor = "Không"
        metaphor_phrases = []
        metaphor_types = []

    # Clean placeholder "[...]"
    if metaphor_phrases == ["[...]"]:
        metaphor_phrases = []
    if metaphor_types == ["[...]"]:
        metaphor_types = []

    return {
        "has_metaphor": has_metaphor,
        "metaphor_phrases": metaphor_phrases,
        "metaphor_types": metaphor_types
    }


# Phân tích
for idx, s in enumerate(sentences, 1):
    print(f"Đang phân tích câu {idx}/{len(sentences)}...")
    res = analyze_metaphor(s)
    print(res)
    parsed = parse_analysis(res)
    results.append({
        "sentence": s,
        "has_metaphor": parsed["has_metaphor"],
        "metaphor_phrases": parsed["metaphor_phrases"],
        "metaphor_types": parsed["metaphor_types"]
    })
    time.sleep(2)

# =====================
# Xuất JSON
# =====================
with open("metaphor_analysis_parsed.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Đã lưu kết quả parse vào 'metaphor_analysis_parsed.json'.")

# =====================
# Xuất CSV
# =====================
df = pd.DataFrame(results)
df.to_csv("metaphor_analysis_results.csv", index=False, encoding="utf-8-sig")
print("Đã lưu kết quả vào 'metaphor_analysis_results.csv'.")

In [ ]:
import pandas as pd
import json

# Đọc file JSON
with open('/kaggle/input/100randomsamples-metaphor-analysis-parsed/100randomsamples_metaphor_analysis_parsed (4) (3).json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Nếu dữ liệu là một danh sách các đối tượng
df = pd.DataFrame(data)

# Lưu thành file CSV
df.to_csv('/kaggle/working/100randomsamples_metaphor_analysis_parsed.csv', index=False, encoding='utf-8-sig')

print("Chuyển đổi thành công!")